# Problem 1 - RNN, LSTM and GRU

Run the setup and validation cells first. The five full experiment cells are intentionally separate so each can be monitored and resumed.

In [1]:
import os
REPO = "/content/BITS_Assignments"
if not os.path.exists(REPO):
    !git clone -q https://github.com/GreedyM0nk/BITS_Assignments.git {REPO}
%cd {REPO}
!git checkout -q 4943872
!git log --oneline -1
%cd "/content/BITS_Assignments/1st_Semester/DNN/2nd Assignment/2024SAMPLE_DNN_Assignment/problem1"

/content/BITS_Assignments
4943872 (HEAD, origin/main, origin/HEAD, main) Strengthen notebook and report evidence workflow
/content/BITS_Assignments/1st_Semester/DNN/2nd Assignment/2024SAMPLE_DNN_Assignment/problem1


In [2]:
from pathlib import Path
import sys
import os
import subprocess
import numpy as np
import torch

ROOT = Path.cwd().parent if Path.cwd().name == "problem1" else Path.cwd()
PROBLEM1 = ROOT / "problem1"
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(PROBLEM1))
os.chdir(ROOT)

from problem1 import (
    run_smoke, run_har, run_adding, run_gradient_study,
    run_hidden_size_sweep, run_clipping_ablation, run_forget_bias_ablation,
)
from src.data import make_adding, validate_adding
from src.models import recurrent_parameter_explanation

print("Repository:", ROOT)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
assert ROOT.name == "2024SAMPLE_DNN_Assignment"


Repository: /content/BITS_Assignments/1st_Semester/DNN/2nd Assignment/2024SAMPLE_DNN_Assignment
PyTorch: 2.14.1+cu130
CUDA: True
GPU: Tesla T4


In [3]:
x, y = make_adding(32, 50, 0)
validate_adding(x, y)
assert np.all(x[:, :, 1].sum(axis=1) == 2)
print(recurrent_parameter_explanation())
print(run_smoke())
assert recurrent_parameter_explanation()["equation_counts"] == {"RNN": 4736, "LSTM": 18944, "GRU": 14208}


{'equation_counts': {'RNN': 4736, 'LSTM': 18944, 'GRU': 14208}, 'pytorch_counts': {'RNN': 4800, 'LSTM': 19200, 'GRU': 14400}, 'text': 'The equation count combines input weights, recurrent weights, and one bias per unit. PyTorch stores separate input-hidden and hidden-hidden biases, adding hidden_size per RNN unit and one extra gate-vector per gated cell.'}
{'device': 'cuda', 'adding_validation': 'passed'}


## Full HAR comparison
Runs three seeds for RNN, LSTM, and GRU and writes rows to `results.csv`.

In [4]:
import time
started = time.perf_counter()
har_results = run_har(False)
print("HAR runtime seconds:", time.perf_counter() - started)
assert len(har_results) == 9


HAR runtime seconds: 40.863138096


## Full adding-problem comparison
Runs all models at T=50, 100, 200, and 400 with three seeds.

In [5]:
started = time.perf_counter()
adding_results = run_adding(False)
print("Adding runtime seconds:", time.perf_counter() - started)
assert len(adding_results) == 36


Adding runtime seconds: 128.474014915


## Gradient study
Runs the T=100 before/after gradient-norm experiment.

In [6]:
started = time.perf_counter()
gradient_results = run_gradient_study(False)
print("Gradient runtime seconds:", time.perf_counter() - started)
assert set(gradient_results) == {"RNN", "LSTM", "GRU"}


Gradient runtime seconds: 9.763524813000004


## Hidden-size sweep
Runs LSTM and GRU at hidden sizes 16, 32, 64, and 128.

In [7]:
started = time.perf_counter()
hidden_results = run_hidden_size_sweep(False)
print("Hidden-size runtime seconds:", time.perf_counter() - started)
assert len(hidden_results) == 24


Hidden-size runtime seconds: 74.42967410599994


## Individual ablation
Runs vanilla-RNN gradient clipping at T=200, norm 1.0 versus none. The
separate forget-bias cell below is available if course staff confirms the
original ID-based ablation rule still applies.

In [8]:
started = time.perf_counter()
clipping_results = run_clipping_ablation(False)
print("Clipping runtime seconds:", time.perf_counter() - started)
assert len(clipping_results) == 6


/usr/local/lib/python3.13/dist-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:178: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:211: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Clipping runtime seconds: 21.150031678000005


In [9]:
# Optional original-rubric ablation; enable only if course staff confirms it.
if False:
    forget_bias_results = run_forget_bias_ablation(False)
    assert len(forget_bias_results) == 6


In [12]:
import os
from google.colab import files

P1 = "/content/BITS_Assignments/1st_Semester/DNN/2nd Assignment/2024SAMPLE_DNN_Assignment/problem1"
os.chdir(P1)

!ls -l figures/p1_adding_T200_learning_curves.png figures/p1_gradient_norms.png
!cd .. && zip -qr /content/problem1_outputs.zip problem1/figures problem1/*.csv

files.download("/content/problem1_outputs.zip")

-rw-r--r-- 1 root root 102073 Nov 12 19:05 figures/p1_adding_T200_learning_curves.png
-rw-r--r-- 1 root root 102711 Nov 12 19:05 figures/p1_gradient_norms.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>